# Meteorology Revisions

Targeted revisions to the meteorology × source apportionment analysis.

| # | Revision | Details |
|---|----------|---------|
| 1 | **Wind Rose Revisions** | 8 sectors, m/s, exclude calm (<2 m/s), consistent outer-ring scale |
| 2 | **RH Threshold Analysis** | Cross-plots at 40 % and 50 % RH for all 3 method pairings |
| 3 | **Temperature Analysis** | T_mean / T_min / T_max vs biomass concentration; fix zero-conc points |
| 4 | **RH × Season × Biomass** | Source concentration vs absolute RH by season (biomass focus) |
| 5 | **Seasonal Correlation Matrices** | Three separate met × source correlation heatmaps by season |

## Section 0: Setup & Data Loading

In [ ]:
import sys
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.projections.polar import PolarAxes
import matplotlib.cm as cm
import warnings
warnings.filterwarnings('ignore')

# Resolve repo paths for portable notebook execution
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / 'pyproject.toml').exists()), cwd)
notebook_dir = str((repo_root / 'notebooks').resolve())
data_root = Path(
    os.environ.get('AETHMODULAR_DATA_ROOT', str(repo_root / 'research' / 'ftir_hips_chem'))
).expanduser().resolve()

# Add research scripts folder to path
scripts_path = str((repo_root / 'research' / 'ftir_hips_chem' / 'scripts').resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from config import SITES, MAC_VALUE
from data_matching import (
    load_aethalometer_data,
    load_filter_data,
    add_base_filter_id,
    match_all_parameters,
    load_etad_factors_with_filter_ids,
)

print(f'Repo root: {repo_root}')
print(f'Data root: {data_root}')
print(f'MAC value: {MAC_VALUE} m\u00b2/g')

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

ADDIS_CONFIG = {
    'name': 'Addis_Ababa',
    'code': 'ETAD',
    'timezone': 'Africa/Addis_Ababa',
}

# Ethiopian seasons
SEASONS = {
    'Dry Season': [10, 11, 12, 1, 2],
    'Belg Rainy Season': [3, 4, 5],
    'Kiremt Rainy Season': [6, 7, 8, 9],
}
SEASONS_ORDER = ['Dry Season', 'Belg Rainy Season', 'Kiremt Rainy Season']
SEASON_COLORS = {
    'Dry Season': '#E67E22',
    'Belg Rainy Season': '#27AE60',
    'Kiremt Rainy Season': '#3498DB',
}

def get_season_3(month):
    for season, months in SEASONS.items():
        if month in months:
            return season
    return None

# Source categories
SOURCE_CATEGORIES = {
    'charcoal':        {'label': 'Charcoal Burning',       'color': '#2C3E50', 'marker': 'o'},
    'wood':            {'label': 'Wood Burning',           'color': '#8B4513', 'marker': 's'},
    'fossil_fuel':     {'label': 'Fossil Fuel Combustion', 'color': '#7D3C98', 'marker': '^'},
    'polluted_marine': {'label': 'Polluted Marine',        'color': '#2980B9', 'marker': 'D'},
    'sea_salt':        {'label': 'Sea Salt Mixed',         'color': '#1ABC9C', 'marker': 'v'},
}
SOURCE_ORDER = ['charcoal', 'wood', 'fossil_fuel', 'polluted_marine', 'sea_salt']

GROUPED_SOURCE_CATEGORIES = {
    'biomass_burning':        {'label': 'Biomass Burning',        'color': '#556B2F', 'marker': 'o'},
    'fossil_fuel_combustion': {'label': 'Fossil Fuel Combustion', 'color': '#8B0000', 'marker': '^'},
    'sea_salt':               {'label': 'Sea Salt Mixed',         'color': '#1ABC9C', 'marker': 'v'},
}
GROUPED_SOURCE_ORDER = ['biomass_burning', 'fossil_fuel_combustion', 'sea_salt']

# Column mappings
FACTOR_TO_FRAC = {
    'GF3 (Charcoal)':              'charcoal_frac',
    'GF2 (Wood Burning)':          'wood_frac',
    'GF5 (Fossil Fuel Combustion)':'fossil_fuel_frac',
    'GF4 (Polluted Marine)':       'polluted_marine_frac',
    'GF1 (Sea Salt Mixed)':        'sea_salt_frac',
}
FACTOR_TO_CONC = {
    'K_F3 Charcoal (ug/m3)':              'charcoal_conc',
    'K_F2 Wood Burning (ug/m3)':           'wood_conc',
    'K_F5 Fossil Fuel Combustion (ug/m3)': 'fossil_fuel_conc',
    'K_F4 Polluted Marine (ug/m3)':        'polluted_marine_conc',
    'K_F1 Sea Salt Mixed (ug/m3)':         'sea_salt_conc',
}

CONC_TO_SOURCE = {
    'charcoal_conc':        'charcoal',
    'wood_conc':            'wood',
    'fossil_fuel_conc':     'fossil_fuel',
    'polluted_marine_conc': 'polluted_marine',
    'sea_salt_conc':        'sea_salt',
}

frac_cols = list(FACTOR_TO_FRAC.values())
conc_cols = list(FACTOR_TO_CONC.values())

# Font configuration
FONT = {'title': 16, 'axis': 14, 'tick': 12, 'legend': 11, 'annot': 11}
plt.rcParams.update({
    'font.size': FONT['tick'],
    'axes.titlesize': FONT['title'],
    'axes.labelsize': FONT['axis'],
    'legend.fontsize': FONT['legend'],
})

# BC/EC method pairings (used throughout)
PAIRS_FOR_RH = [
    ('ftir_ec', 'hips_fabs', 'FTIR EC (\u00b5g/m\u00b3)', 'HIPS Fabs/MAC (\u00b5g/m\u00b3)'),
    ('ftir_ec', 'ir_bcc',    'FTIR EC (\u00b5g/m\u00b3)', 'Aeth IR BCc (\u00b5g/m\u00b3)'),
    ('hips_fabs', 'ir_bcc',  'HIPS Fabs/MAC (\u00b5g/m\u00b3)', 'Aeth IR BCc (\u00b5g/m\u00b3)'),
]

# Output directories
output_root = repo_root / 'artifacts' / 'notebook_outputs' / 'meteorology_revisions'
dirs = {}
for d in ['plots', 'data']:
    path = output_root / d
    path.mkdir(parents=True, exist_ok=True)
    dirs[d] = str(path)

print(f'Configuration ready. Outputs: {output_root}')

In [ ]:
# =============================================================================
# Load Meteorological Data  (wspd converted to m/s)
# =============================================================================

def resolve_met_path(*filenames):
    search_dirs = [
        data_root / 'Weather Data' / 'Meteostat',
        Path(notebook_dir),
    ]
    for base in search_dirs:
        for name in filenames:
            candidate = base / name
            if candidate.exists():
                return candidate
    search_text = ', '.join(str(d) for d in search_dirs)
    files_text = ', '.join(filenames)
    raise FileNotFoundError(f'Could not find [{files_text}] under: {search_text}')


# --- Daily met data ---
MET_DAILY_PATH = resolve_met_path(
    'Addis_Ababa_daily_Average_met_Data.csv',
    'Addis Ababa daily Average met Data.csv',
)
met_daily = pd.read_csv(MET_DAILY_PATH, encoding='utf-8-sig')
met_daily['date'] = pd.to_datetime(met_daily['date'])
met_daily['date_only'] = met_daily['date'].dt.normalize()

# Convert wind speed from km/h to m/s
met_daily['wspd'] = met_daily['wspd'] / 3.6

# Derived columns
met_daily['temp_range'] = met_daily['tmax'] - met_daily['tmin']
met_daily['Month'] = met_daily['date'].dt.month
met_daily['season'] = met_daily['Month'].apply(get_season_3)

print(f'Daily met file: {MET_DAILY_PATH}')
print(f"Daily met: {len(met_daily)} days, {met_daily['date'].min().date()} to {met_daily['date'].max().date()}")
print(f"  Columns: {list(met_daily.columns)}")
print(f"  Wind speed range: {met_daily['wspd'].min():.1f} \u2013 {met_daily['wspd'].max():.1f} m/s")

# --- Hourly met data ---
MET_HOURLY_PATH = resolve_met_path('master_meteostat_AddisAbaba_63450_2022-12-01_2024-10-01.csv')
met_hourly = pd.read_csv(MET_HOURLY_PATH)
met_hourly = met_hourly.replace('NA', np.nan)
met_hourly['time'] = pd.to_datetime(met_hourly['time'])

for col in ['temp', 'dwpt', 'rhum', 'prcp', 'wdir', 'wspd', 'pres']:
    met_hourly[col] = pd.to_numeric(met_hourly[col], errors='coerce')

# Convert hourly wind speed from km/h to m/s
met_hourly['wspd'] = met_hourly['wspd'] / 3.6

met_hourly['date_only'] = met_hourly['time'].dt.normalize()
met_hourly['hour'] = met_hourly['time'].dt.hour
met_hourly['Month'] = met_hourly['time'].dt.month
met_hourly['season'] = met_hourly['Month'].apply(get_season_3)

# Daily aggregates from hourly
hourly_daily_agg = met_hourly.groupby('date_only').agg(
    rhum_mean=('rhum', 'mean'),
    rhum_max=('rhum', 'max'),
    rhum_min=('rhum', 'min'),
    dwpt_mean=('dwpt', 'mean'),
    wdir_mode=('wdir', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan),
    wdir_mean=('wdir', 'mean'),
    wspd_mean=('wspd', 'mean'),
    wspd_max=('wspd', 'max'),
).reset_index()

print(f'\nHourly met: {len(met_hourly)} records')
print(f"  Wind speed range: {met_hourly['wspd'].min():.1f} \u2013 {met_hourly['wspd'].max():.1f} m/s")
print(f"  Calm observations (< 2 m/s): {(met_hourly['wspd'] < 2).sum()} / {met_hourly['wspd'].notna().sum()}")
print(f'  Daily aggregates: {len(hourly_daily_agg)} days')

In [ ]:
# =============================================================================
# Load Source Apportionment + BC/EC Data
# =============================================================================

factors_df = load_etad_factors_with_filter_ids()
factors_df = factors_df.rename(columns={**FACTOR_TO_FRAC, **FACTOR_TO_CONC})

aethalometer_data = load_aethalometer_data()
filter_data = load_filter_data()
filter_data = add_base_filter_id(filter_data)

df_aeth = aethalometer_data.get('Addis_Ababa')
bc_df = match_all_parameters('Addis_Ababa', 'ETAD', df_aeth, filter_data)

etad_filters = filter_data[filter_data['Site'] == 'ETAD'][['SampleDate', 'FilterId']].drop_duplicates()
etad_filters = etad_filters.rename(columns={'SampleDate': 'date', 'FilterId': 'base_filter_id'})
bc_df['date'] = pd.to_datetime(bc_df['date'])
etad_filters['date'] = pd.to_datetime(etad_filters['date'])
bc_with_id = pd.merge(bc_df, etad_filters, on='date', how='left')

factor_merge_cols = ['base_filter_id'] + frac_cols + conc_cols
available_merge_cols = [c for c in factor_merge_cols if c in factors_df.columns]
df = pd.merge(bc_with_id, factors_df[available_merge_cols], on='base_filter_id', how='left')

df['Month'] = df['date'].dt.month
df['season'] = df['Month'].apply(get_season_3)
df['date_only'] = df['date'].dt.normalize()

available_conc = [c for c in conc_cols if c in df.columns]
if available_conc:
    df['dominant_conc_col'] = df[available_conc].idxmax(axis=1)
    df['dominant_source'] = df['dominant_conc_col'].map(CONC_TO_SOURCE)
    df['kf_total'] = df[available_conc].sum(axis=1)
    df['dominant_kf_value'] = df[available_conc].max(axis=1)
    df['dominant_fraction'] = df['dominant_kf_value'] / df['kf_total']

if 'charcoal_conc' in df.columns and 'wood_conc' in df.columns:
    df['biomass_conc'] = df['charcoal_conc'].fillna(0) + df['wood_conc'].fillna(0)
if 'fossil_fuel_conc' in df.columns and 'polluted_marine_conc' in df.columns:
    df['fossil_comb_conc'] = df['fossil_fuel_conc'].fillna(0) + df['polluted_marine_conc'].fillna(0)

print(f"Source-apportioned dataset: {len(df)} samples")
print(f"Dominant source distribution:\n{df['dominant_source'].value_counts()}")

In [ ]:
# =============================================================================
# Merge Met Data with Source Apportionment Data
# =============================================================================

met_merge_cols = ['date_only', 'tavg', 'tmin', 'tmax', 'prcp', 'wspd', 'pres', 'temp_range']
met_merge = met_daily[[c for c in met_merge_cols if c in met_daily.columns]].copy()
df = pd.merge(df, met_merge, on='date_only', how='left', suffixes=('', '_met'))
df = pd.merge(df, hourly_daily_agg, on='date_only', how='left')

met_matched = df['tavg'].notna().sum()
rh_matched = df['rhum_mean'].notna().sum()

print(f"Met data matched to source samples:")
print(f"  Temperature: {met_matched}/{len(df)} ({met_matched/len(df)*100:.0f}%)")
print(f"  Humidity:    {rh_matched}/{len(df)} ({rh_matched/len(df)*100:.0f}%)")
print(f"  Wind dir:    {df['wdir_mean'].notna().sum()}/{len(df)}")
print(f"  Precip:      {df['prcp'].notna().sum()}/{len(df)}")

---

# Revision 1: Wind Rose Revisions

- **8 direction sectors** centered on N / NE / E / SE / S / SW / W / NW (±22.5°)
- **Calm winds excluded** (wspd < 2 m/s)
- **Speed units in m/s**
- **Consistent outer-ring scale** across all seasonal panels

---

In [ ]:
# =============================================================================
# Wind Rose Helper — 8 sectors, m/s, calm exclusion
# =============================================================================

def wind_rose(ax, directions, speeds=None, n_bins=8, title='',
              color='steelblue', speed_bins=None, calm_threshold=2.0,
              rlim=None):
    """
    Draw a wind rose on a polar axis.

    Parameters
    ----------
    ax : polar axes
    directions : array of wind directions in degrees
    speeds : array of wind speeds in m/s (for speed-binned coloring)
    n_bins : number of direction bins (8 = cardinal + intercardinal)
    calm_threshold : exclude winds below this speed (m/s). Set to 0 to keep all.
    rlim : fixed outer-ring scale (percent). If None, auto-scale.
    """
    # Align bin edges so that bin centres sit on 0, 45, 90 …
    half_bin = 360 / n_bins / 2  # 22.5° for 8 bins
    dir_bins = np.linspace(-half_bin, 360 - half_bin, n_bins + 1)
    bin_width = 2 * np.pi / n_bins

    valid = np.isfinite(directions)
    dirs = np.array(directions)[valid]

    # Apply calm-wind filter
    n_total = len(dirs)
    if speeds is not None and calm_threshold > 0:
        spds = np.array(speeds)[valid]
        calm_mask = spds >= calm_threshold
        dirs = dirs[calm_mask]
        spds = spds[calm_mask]
    elif speeds is not None:
        spds = np.array(speeds)[valid]
    else:
        spds = None

    n_plotted = len(dirs)
    n_calm = n_total - n_plotted

    # Wrap directions so the first bin (centred on N=0°) works
    dirs_wrapped = np.where(dirs >= 360 - half_bin, dirs - 360, dirs)

    if spds is not None and speed_bins is not None:
        colors_speed = plt.cm.YlOrRd(np.linspace(0.2, 0.9, len(speed_bins) - 1))
        bottom = np.zeros(n_bins)

        for j in range(len(speed_bins) - 1):
            spd_mask = (spds >= speed_bins[j]) & (spds < speed_bins[j + 1])
            counts, _ = np.histogram(dirs_wrapped[spd_mask], bins=dir_bins)
            freq = counts / n_plotted * 100  # percent of non-calm obs
            theta = np.deg2rad(np.arange(0, 360, 360 / n_bins))  # bin centres
            ax.bar(theta, freq, width=bin_width, bottom=bottom,
                   color=colors_speed[j], edgecolor='white', linewidth=0.5,
                   label=f'{speed_bins[j]}–{speed_bins[j+1]} m/s')
            bottom += freq
    else:
        counts, _ = np.histogram(dirs_wrapped, bins=dir_bins)
        freq = counts / n_plotted * 100 if n_plotted > 0 else counts
        theta = np.deg2rad(np.arange(0, 360, 360 / n_bins))
        ax.bar(theta, freq, width=bin_width, color=color,
               edgecolor='white', linewidth=0.5, alpha=0.8)

    # Consistent outer-ring scale
    if rlim is not None:
        ax.set_rlim(0, rlim)

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315],
                       ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW'])
    ax.set_title(title, fontsize=FONT['title'], fontweight='bold', pad=20)

    # Annotate calm count
    if n_calm > 0:
        pct_calm = n_calm / n_total * 100
        ax.text(0.5, -0.08, f'Calm (<{calm_threshold} m/s): {n_calm} obs ({pct_calm:.1f}%)',
                transform=ax.transAxes, ha='center', fontsize=FONT['annot'] - 1,
                style='italic')

    return n_plotted, n_calm

print('Wind rose function defined (8-sector, m/s, calm-excluded)')

In [ ]:
# =============================================================================
# 1A: Overall Wind Rose — 8 sectors, m/s, calm excluded
# =============================================================================

SPEED_BINS_MS = [2, 4, 6, 8, 50]  # m/s bins (calm < 2 already excluded)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='polar')

valid_hourly = met_hourly.dropna(subset=['wdir', 'wspd'])
n_plot, n_calm = wind_rose(
    ax, valid_hourly['wdir'].values, speeds=valid_hourly['wspd'].values,
    speed_bins=SPEED_BINS_MS, calm_threshold=2.0,
    title=f'Addis Ababa Wind Rose\n(n={len(valid_hourly):,} hourly obs, calm excluded)'
)
ax.legend(loc='lower right', bbox_to_anchor=(1.3, -0.05), fontsize=FONT['legend'])

plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'wind_rose_overall_8sect.png'),
            dpi=200, bbox_inches='tight')
plt.show()
print(f'Plotted {n_plot} observations; {n_calm} calm observations excluded')

In [ ]:
# =============================================================================
# 1B: Wind Rose by Dominant Source Type — 8 sectors, calm excluded
# =============================================================================

df_wind = df.dropna(subset=['wdir_mean', 'dominant_source']).copy()
sources_to_plot = [s for s in SOURCE_ORDER if s != 'sea_salt']

fig, axes = plt.subplots(1, 4, figsize=(20, 5),
                          subplot_kw=dict(projection='polar'))

for idx, source in enumerate(sources_to_plot):
    ax = axes[idx]
    info = SOURCE_CATEGORIES[source]
    source_dates = df_wind[df_wind['dominant_source'] == source]['date_only'].values
    hourly_subset = met_hourly[met_hourly['date_only'].isin(source_dates)]
    hourly_valid = hourly_subset.dropna(subset=['wdir', 'wspd'])

    if len(hourly_valid) > 10:
        wind_rose(ax, hourly_valid['wdir'].values,
                  speeds=hourly_valid['wspd'].values,
                  calm_threshold=2.0,
                  rlim=60,
                  title=f"{info['label']}\n(n={len(source_dates)} days)",
                  color=info['color'])
    else:
        ax.set_title(f"{info['label']}\n(insufficient data)",
                    fontsize=FONT['title'])

plt.suptitle('Wind Direction on Days Dominated by Each Source (calm < 2 m/s excluded)',
             fontsize=FONT['title'] + 2, fontweight='bold', y=1.08)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'wind_rose_by_source_8sect.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 1C: Seasonal Wind Roses — consistent outer-ring scale
# =============================================================================

# Pre-compute the maximum frequency across all seasons to set a shared rlim
n_bins = 8
half_bin = 360 / n_bins / 2
dir_bin_edges = np.linspace(-half_bin, 360 - half_bin, n_bins + 1)

max_freq = 0
for season in SEASONS_ORDER:
    sdata = met_hourly[(met_hourly['season'] == season)].dropna(subset=['wdir', 'wspd'])
    sdata = sdata[sdata['wspd'] >= 2.0]  # exclude calm
    if len(sdata) == 0:
        continue
    dirs_w = np.where(sdata['wdir'].values >= 360 - half_bin,
                      sdata['wdir'].values - 360, sdata['wdir'].values)
    counts, _ = np.histogram(dirs_w, bins=dir_bin_edges)
    freq = counts / len(sdata) * 100
    max_freq = max(max_freq, freq.max())

# Round up to nearest 5 for a clean scale
shared_rlim = int(np.ceil(max_freq / 5) * 5)
print(f'Shared outer-ring scale: {shared_rlim}%')

fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                          subplot_kw=dict(projection='polar'))

for idx, season in enumerate(SEASONS_ORDER):
    ax = axes[idx]
    season_data = met_hourly[met_hourly['season'] == season].dropna(subset=['wdir', 'wspd'])

    n_plot, n_calm = wind_rose(
        ax, season_data['wdir'].values, speeds=season_data['wspd'].values,
        speed_bins=SPEED_BINS_MS, calm_threshold=2.0,
        rlim=shared_rlim,
        title=f"{season}\n(n={len(season_data):,})")

axes[-1].legend(loc='lower right', bbox_to_anchor=(1.4, -0.05), fontsize=FONT['legend'])
plt.suptitle('Seasonal Wind Patterns \u2014 Addis Ababa (8 sectors, calm < 2 m/s excluded)',
             fontsize=FONT['title'] + 2, fontweight='bold', y=1.08)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'wind_rose_seasonal_8sect.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

# Revision 2: RH Threshold Analysis (40 % and 50 %)

Cross-plots for all three method pairings at **40 %** and **50 %** RH thresholds, with sample sizes shown for each low-RH subset.

---

In [ ]:
# =============================================================================
# 2: RH Threshold Cross-Plots at 40% and 50%
# =============================================================================

df_rh = df.dropna(subset=['rhum_mean']).copy()
RH_THRESHOLDS = [40, 50]

rh_colors_split = {'Low': '#E67E22', 'High': '#3498DB'}

for rh_thresh in RH_THRESHOLDS:
    df_rh[f'rh_cat_{rh_thresh}'] = np.where(
        df_rh['rhum_mean'] <= rh_thresh, 'Low', 'High')

    n_low = (df_rh[f'rh_cat_{rh_thresh}'] == 'Low').sum()
    n_high = (df_rh[f'rh_cat_{rh_thresh}'] == 'High').sum()
    print(f'\nRH threshold {rh_thresh}%: Low-RH n={n_low}, High-RH n={n_high}')

    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    for col_idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
        # Consistent axis limit for this pair
        pair_data = df_rh.dropna(subset=[x_col, y_col])
        if len(pair_data) == 0:
            continue
        all_vals = np.concatenate([pair_data[x_col].values, pair_data[y_col].values])
        lim = np.nanpercentile(all_vals, 99) * 1.1

        for row_idx, rh_cat in enumerate(['Low', 'High']):
            ax = axes[row_idx, col_idx]
            subset = df_rh[df_rh[f'rh_cat_{rh_thresh}'] == rh_cat].dropna(
                subset=[x_col, y_col])

            if len(subset) < 5:
                ax.text(0.5, 0.5, f'{rh_cat}-RH\nInsufficient data',
                        transform=ax.transAxes, ha='center')
                continue

            ax.scatter(subset[x_col], subset[y_col],
                      c=rh_colors_split[rh_cat], s=50, alpha=0.7,
                      edgecolors='black', linewidths=0.5)

            slope, intercept, r, p, _ = stats.linregress(subset[x_col], subset[y_col])
            x_line = np.linspace(0, lim, 100)
            ax.plot(x_line, slope * x_line + intercept,
                   color=rh_colors_split[rh_cat], linewidth=2, alpha=0.8)
            ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, linewidth=1)

            rh_range = subset['rhum_mean']
            sign = '+' if intercept >= 0 else '-'
            ax.text(0.05, 0.95,
                    f'{rh_cat}-RH (RH: {rh_range.min():.0f}\u2013{rh_range.max():.0f}%)\n'
                    f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                    f'R\u00b2 = {r**2:.3f}, n = {len(subset)}',
                    transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                    bbox=dict(boxstyle='round',
                              facecolor=rh_colors_split[rh_cat], alpha=0.2))

            ax.set_xlabel(x_label, fontsize=FONT['axis'] - 1)
            ax.set_ylabel(y_label, fontsize=FONT['axis'] - 1)
            ax.set_xlim(0, lim)
            ax.set_ylim(0, lim)
            ax.set_aspect('equal', adjustable='box')
            ax.grid(True, alpha=0.3)

            if col_idx == 0:
                label_rh = f'\u2264 {rh_thresh}%' if rh_cat == 'Low' else f'> {rh_thresh}%'
                ax.set_ylabel(f'RH {label_rh}\n{y_label}', fontsize=FONT['axis'])

    plt.suptitle(
        f'BC/EC Measurement Agreement: RH \u2264 {rh_thresh}% (n={n_low}) vs '
        f'RH > {rh_thresh}% (n={n_high})',
        fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(dirs['plots'],
                f'rh_threshold_{rh_thresh}pct_crossplots.png'),
                dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# 2B: Summary table — slopes & R\u00b2 at each RH threshold
# =============================================================================

print('=' * 90)
print('RH THRESHOLD REGRESSION SUMMARY')
print('=' * 90)

for rh_thresh in RH_THRESHOLDS:
    print(f'\n--- RH threshold: {rh_thresh}% ---')
    print(f'{"Pair":<35s} {"Subset":<10s} {"n":>5s} {"Slope":>8s} {"R\u00b2":>8s} {"p":>8s}')
    print('-' * 80)

    for x_col, y_col, x_label, y_label in PAIRS_FOR_RH:
        pair_name = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
        for rh_cat in ['Low', 'High']:
            subset = df_rh[df_rh[f'rh_cat_{rh_thresh}'] == rh_cat].dropna(
                subset=[x_col, y_col])
            if len(subset) >= 5:
                s, i, r, p, _ = stats.linregress(subset[x_col], subset[y_col])
                label_rh = f'\u2264{rh_thresh}%' if rh_cat == 'Low' else f'>{rh_thresh}%'
                print(f'{pair_name:<35s} {label_rh:<10s} {len(subset):5d} '
                      f'{s:8.3f} {r**2:8.3f} {p:8.4f}')
            else:
                print(f'{pair_name:<35s} {rh_cat:<10s}   <5 samples')

---

# Revision 2C: Expanded RH Threshold Sweep

Deep-dive into how measurement agreement changes with humidity:

| Sub-section | Analysis |
|-------------|----------|
| **2C-i** | Threshold sweep with **sample-size annotations** at each threshold |
| **2C-ii** | **Slope-difference significance** (are dry/humid slopes statistically different?) |
| **2C-iii** | **Optimal threshold identification** (maximize slope divergence) |
| **2C-iv** | **Cross-plots at optimal threshold** for each method pairing |
| **2C-v** | **Season-stratified threshold sweep** (does the pattern hold within each season?) |

---

In [ ]:
# =============================================================================
# 2C-i: RH Threshold Sweep with Sample-Size & Split-Ratio Panels
# =============================================================================
# Ideal slope = 1.0 (perfect 1:1 agreement between methods).
# A slope further from 1 means the methods diverge more.

rh_thresholds = np.arange(30, 90, 5)

fig, axes = plt.subplots(3, 3, figsize=(18, 14),
                          gridspec_kw={'height_ratios': [2, 1, 1]})

# Store sweep results for later use
sweep_results = {}

for idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
    ax_slope = axes[0, idx]
    ax_dist  = axes[1, idx]
    ax_n     = axes[2, idx]
    pair_key = f'{y_col}_vs_{x_col}'

    slopes_below, slopes_above = [], []
    r2_below, r2_above = [], []
    n_below_list, n_above_list = [], []

    for rh_thresh in rh_thresholds:
        below = df_rh[df_rh['rhum_mean'] <= rh_thresh].dropna(subset=[x_col, y_col])
        above = df_rh[df_rh['rhum_mean'] > rh_thresh].dropna(subset=[x_col, y_col])
        n_below_list.append(len(below))
        n_above_list.append(len(above))

        if len(below) >= 5:
            s, _, r, _, _ = stats.linregress(below[x_col], below[y_col])
            slopes_below.append(s)
            r2_below.append(r**2)
        else:
            slopes_below.append(np.nan)
            r2_below.append(np.nan)

        if len(above) >= 5:
            s, _, r, _, _ = stats.linregress(above[x_col], above[y_col])
            slopes_above.append(s)
            r2_above.append(r**2)
        else:
            slopes_above.append(np.nan)
            r2_above.append(np.nan)

    sweep_results[pair_key] = {
        'slopes_below': slopes_below, 'slopes_above': slopes_above,
        'r2_below': r2_below, 'r2_above': r2_above,
        'n_below': n_below_list, 'n_above': n_above_list,
    }

    n_below_arr = np.array(n_below_list)
    n_above_arr = np.array(n_above_list)
    n_total_arr = n_below_arr + n_above_arr
    frac_low = np.where(n_total_arr > 0, n_below_arr / n_total_arr * 100, 0)

    s_below_arr = np.array(slopes_below)
    s_above_arr = np.array(slopes_above)
    dist_below = np.abs(s_below_arr - 1.0)
    dist_above = np.abs(s_above_arr - 1.0)

    # --- Row 1: slopes + ideal line ---
    ax_slope.axhline(y=1.0, color='green', linewidth=2.5, alpha=0.6,
                     label='Ideal (slope = 1)')
    ax_slope.axhspan(0.9, 1.1, color='green', alpha=0.07)
    ax_slope.plot(rh_thresholds, slopes_below, 'o-', color='#E67E22', linewidth=2,
                  markersize=6, label='Low-RH (RH $\leq$ threshold)')
    ax_slope.plot(rh_thresholds, slopes_above, 's-', color='#3498DB', linewidth=2,
                  markersize=6, label='High-RH (RH > threshold)')

    ax2 = ax_slope.twinx()
    ax2.plot(rh_thresholds, r2_below, 'o--', color='#E67E22', linewidth=1,
             markersize=4, alpha=0.4)
    ax2.plot(rh_thresholds, r2_above, 's--', color='#3498DB', linewidth=1,
             markersize=4, alpha=0.4)
    ax2.set_ylabel('R$^2$ (dashed)', fontsize=FONT['axis'] - 2, alpha=0.6)
    ax2.set_ylim(0, 1)

    pair_title = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
    ax_slope.set_title(pair_title, fontsize=FONT['title'])
    ax_slope.set_ylabel('Regression Slope', fontsize=FONT['axis'])
    ax_slope.legend(fontsize=FONT['legend'] - 1, loc='upper left')
    ax_slope.grid(True, alpha=0.3)
    ax_slope.set_xticklabels([])

    # --- Row 2: |slope - 1| distance from ideal ---
    ax_dist.fill_between(rh_thresholds, dist_below, alpha=0.3, color='#E67E22')
    ax_dist.fill_between(rh_thresholds, dist_above, alpha=0.3, color='#3498DB')
    ax_dist.plot(rh_thresholds, dist_below, 'o-', color='#E67E22', linewidth=2,
                 markersize=5, label='Low-RH')
    ax_dist.plot(rh_thresholds, dist_above, 's-', color='#3498DB', linewidth=2,
                 markersize=5, label='High-RH')
    ax_dist.axhline(y=0, color='green', linewidth=1.5, alpha=0.5)
    ax_dist.set_ylabel('|slope $-$ 1|', fontsize=FONT['axis'] - 1)
    ax_dist.legend(fontsize=FONT['legend'] - 2, loc='upper right', ncol=2)
    ax_dist.grid(True, alpha=0.3)
    ax_dist.set_xticklabels([])

    # Annotate the threshold where low-RH is closest to ideal
    valid_lo = ~np.isnan(dist_below)
    if valid_lo.any():
        best_lo_idx = np.nanargmin(dist_below)
        ax_dist.annotate(
            f'Best low-RH:\n{rh_thresholds[best_lo_idx]}% '
            f'(|s-1|={dist_below[best_lo_idx]:.3f})',
            xy=(rh_thresholds[best_lo_idx], dist_below[best_lo_idx]),
            xytext=(rh_thresholds[best_lo_idx] + 8, dist_below[best_lo_idx] + 0.05),
            fontsize=8, color='#E67E22', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#E67E22', lw=1.5))

    # --- Row 3: stacked sample counts + ratio ---
    ax_n.bar(rh_thresholds, n_below_arr, width=3.5, color='#E67E22',
             alpha=0.7, label='n (low-RH)')
    ax_n.bar(rh_thresholds, n_above_arr, width=3.5, bottom=n_below_arr,
             color='#3498DB', alpha=0.7, label='n (high-RH)')

    for i, rh_t in enumerate(rh_thresholds):
        if n_total_arr[i] > 0:
            ax_n.text(rh_t, n_total_arr[i] + 1,
                      f'{frac_low[i]:.0f}%', ha='center', fontsize=8,
                      fontweight='bold', color='#E67E22')

    ax_n.set_xlabel('RH Threshold (%)', fontsize=FONT['axis'])
    ax_n.set_ylabel('Sample Count', fontsize=FONT['axis'] - 1)
    ax_n.legend(fontsize=FONT['legend'] - 2, loc='upper left', ncol=2)
    ax_n.grid(True, axis='y', alpha=0.3)

    ax_r = ax_n.twinx()
    ax_r.plot(rh_thresholds, frac_low, 'D-', color='#8E44AD', linewidth=1.5,
              markersize=4, alpha=0.8)
    ax_r.set_ylabel('% low-RH', fontsize=FONT['axis'] - 2, color='#8E44AD')
    ax_r.set_ylim(0, 100)
    ax_r.tick_params(axis='y', labelcolor='#8E44AD')

plt.suptitle('RH Threshold Sweep: Slopes, Distance from Ideal (1:1), and Sample Split',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'rh_sweep_with_sample_sizes.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 2C-ii: Slope-Difference Significance Test at Each Threshold
# =============================================================================
# Test whether the low-RH and high-RH slopes are statistically different
# using the Chow-test-like approach: compare SE of slope difference.

print('=' * 100)
print('SLOPE DIFFERENCE SIGNIFICANCE TEST (low-RH vs high-RH)')
print('H0: slopes are equal;  reject if p < 0.05')
print('=' * 100)

def slope_difference_test(x1, y1, x2, y2):
    """Test whether two regression slopes are significantly different."""
    n1, n2 = len(x1), len(x2)
    s1, i1, r1, _, se1 = stats.linregress(x1, y1)
    s2, i2, r2, _, se2 = stats.linregress(x2, y2)

    # Standard errors of the slopes
    # se from linregress is SE of the slope
    se_diff = np.sqrt(se1**2 + se2**2)
    t_stat = (s1 - s2) / se_diff if se_diff > 0 else np.nan
    df_approx = n1 + n2 - 4
    p_val = 2 * stats.t.sf(abs(t_stat), df_approx) if not np.isnan(t_stat) else np.nan
    return s1, s2, se_diff, t_stat, p_val

slope_diff_results = {}

for x_col, y_col, x_label, y_label in PAIRS_FOR_RH:
    pair_name = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
    print(f'\n--- {pair_name} ---')
    print(f'{"RH Thresh":>10s} {"Slope_low":>10s} {"Slope_high":>11s} '
          f'{"Diff":>8s} {"t-stat":>8s} {"p-value":>8s} {"n_low":>6s} {"n_high":>7s}  Sig?')
    print('-' * 90)

    pair_key = f'{y_col}_vs_{x_col}'
    sig_thresholds = []

    for rh_thresh in rh_thresholds:
        below = df_rh[df_rh['rhum_mean'] <= rh_thresh].dropna(subset=[x_col, y_col])
        above = df_rh[df_rh['rhum_mean'] > rh_thresh].dropna(subset=[x_col, y_col])

        if len(below) >= 10 and len(above) >= 10:
            s1, s2, se_d, t, p = slope_difference_test(
                below[x_col].values, below[y_col].values,
                above[x_col].values, above[y_col].values)
            sig = '*' if p < 0.05 else ('.' if p < 0.10 else '')
            print(f'{rh_thresh:10.0f} {s1:10.3f} {s2:11.3f} '
                  f'{s1-s2:8.3f} {t:8.2f} {p:8.4f} {len(below):6d} {len(above):7d}  {sig}')
            if p < 0.05:
                sig_thresholds.append(rh_thresh)
        else:
            print(f'{rh_thresh:10.0f}   (insufficient data in one or both subsets)')

    slope_diff_results[pair_key] = sig_thresholds
    if sig_thresholds:
        print(f'\n  >> Significant slope difference at RH thresholds: {sig_thresholds}')
    else:
        print(f'\n  >> No significant slope differences found at any threshold.')

In [ ]:
# =============================================================================
# 2C-iii: Optimal Threshold Identification (max |slope_low - slope_above|)
# =============================================================================

print('=' * 80)
print('OPTIMAL RH THRESHOLD — Maximum Slope Divergence')
print('=' * 80)

optimal_thresholds = {}

for x_col, y_col, x_label, y_label in PAIRS_FOR_RH:
    pair_key = f'{y_col}_vs_{x_col}'
    pair_name = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
    res = sweep_results[pair_key]

    s_below = np.array(res['slopes_below'])
    s_above = np.array(res['slopes_above'])
    n_below = np.array(res['n_below'])
    n_above = np.array(res['n_above'])

    # Only consider thresholds where both subsets have >= 10 samples
    valid = (~np.isnan(s_below)) & (~np.isnan(s_above)) & (n_below >= 10) & (n_above >= 10)
    if not valid.any():
        print(f'\n{pair_name}: insufficient data at all thresholds')
        optimal_thresholds[pair_key] = np.nan
        continue

    divergence = np.abs(s_below - s_above)
    divergence[~valid] = np.nan
    best_idx = np.nanargmax(divergence)
    best_rh = rh_thresholds[best_idx]
    optimal_thresholds[pair_key] = best_rh

    print(f'\n{pair_name}:')
    print(f'  Optimal threshold: RH = {best_rh}%')
    print(f'  Slope (low-RH):  {s_below[best_idx]:.3f}  (n={n_below[best_idx]})')
    print(f'  Slope (high-RH): {s_above[best_idx]:.3f}  (n={n_above[best_idx]})')
    print(f'  Divergence: |{s_below[best_idx]:.3f} - {s_above[best_idx]:.3f}| = {divergence[best_idx]:.3f}')

# Visualize divergence curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
    ax = axes[idx]
    pair_key = f'{y_col}_vs_{x_col}'
    res = sweep_results[pair_key]

    s_below = np.array(res['slopes_below'])
    s_above = np.array(res['slopes_above'])
    divergence = np.abs(s_below - s_above)

    ax.plot(rh_thresholds, divergence, 'D-', color='#8E44AD', linewidth=2, markersize=6)
    best_rh = optimal_thresholds[pair_key]
    if not np.isnan(best_rh):
        best_div = divergence[list(rh_thresholds).index(best_rh)]
        ax.axvline(x=best_rh, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.annotate(f'Optimal: {best_rh}%\n|diff| = {best_div:.3f}',
                   xy=(best_rh, best_div), xytext=(best_rh + 5, best_div * 0.9),
                   fontsize=FONT['annot'], fontweight='bold',
                   arrowprops=dict(arrowstyle='->', color='red'),
                   bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

    pair_title = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
    ax.set_title(pair_title, fontsize=FONT['title'])
    ax.set_xlabel('RH Threshold (%)', fontsize=FONT['axis'])
    ax.set_ylabel('|Slope$_{low}$ - Slope$_{high}$|', fontsize=FONT['axis'])
    ax.grid(True, alpha=0.3)

plt.suptitle('Slope Divergence vs RH Threshold — Identifying Optimal Split Point',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'rh_optimal_threshold_divergence.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 2C-iv: Cross-Plots at Optimal Threshold for Each Pairing
# =============================================================================

rh_colors_opt = {'Low': '#E67E22', 'High': '#3498DB'}

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

for col_idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
    pair_key = f'{y_col}_vs_{x_col}'
    opt_rh = optimal_thresholds.get(pair_key, 55)
    if np.isnan(opt_rh):
        opt_rh = 55  # fallback

    df_rh[f'rh_opt_{pair_key}'] = np.where(
        df_rh['rhum_mean'] <= opt_rh, 'Low', 'High')

    # Consistent axis limit
    pair_data = df_rh.dropna(subset=[x_col, y_col])
    all_vals = np.concatenate([pair_data[x_col].values, pair_data[y_col].values])
    lim = np.nanpercentile(all_vals, 99) * 1.1

    for row_idx, rh_cat in enumerate(['Low', 'High']):
        ax = axes[row_idx, col_idx]
        subset = df_rh[df_rh[f'rh_opt_{pair_key}'] == rh_cat].dropna(
            subset=[x_col, y_col])

        if len(subset) < 5:
            ax.text(0.5, 0.5, f'{rh_cat}-RH\nInsufficient data',
                    transform=ax.transAxes, ha='center')
            continue

        ax.scatter(subset[x_col], subset[y_col],
                  c=rh_colors_opt[rh_cat], s=50, alpha=0.7,
                  edgecolors='black', linewidths=0.5)

        slope, intercept, r, p, _ = stats.linregress(subset[x_col], subset[y_col])
        x_line = np.linspace(0, lim, 100)
        ax.plot(x_line, slope * x_line + intercept,
               color=rh_colors_opt[rh_cat], linewidth=2, alpha=0.8)
        ax.plot([0, lim], [0, lim], 'k--', alpha=0.4, linewidth=1)

        rh_range = subset['rhum_mean']
        sign = '+' if intercept >= 0 else '-'
        label_rh = f'\u2264 {opt_rh:.0f}%' if rh_cat == 'Low' else f'> {opt_rh:.0f}%'
        ax.text(0.05, 0.95,
                f'RH {label_rh} (range: {rh_range.min():.0f}\u2013{rh_range.max():.0f}%)\n'
                f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, n = {len(subset)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'],
                bbox=dict(boxstyle='round', facecolor=rh_colors_opt[rh_cat], alpha=0.2))

        ax.set_xlabel(x_label, fontsize=FONT['axis'] - 1)
        ax.set_ylabel(y_label, fontsize=FONT['axis'] - 1)
        ax.set_xlim(0, lim)
        ax.set_ylim(0, lim)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.3)

        if col_idx == 0:
            ax.set_ylabel(f'{rh_cat.upper()}-RH\n{y_label}', fontsize=FONT['axis'])

# Build suptitle with per-pair thresholds
thresh_str = ', '.join(
    f'{y.split("(")[0].strip()} vs {x.split("(")[0].strip()}: '
    f'{optimal_thresholds.get(f"{y_col}_vs_{x_col}", 55):.0f}%'
    for x_col, y_col, x, y in PAIRS_FOR_RH)
plt.suptitle(
    f'BC/EC Agreement at Optimal RH Thresholds\n({thresh_str})',
    fontsize=FONT['title'], fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'rh_optimal_threshold_crossplots.png'),
            dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# 2C-v: Season-Stratified RH Threshold Sweep
# =============================================================================
# Does the RH-slope pattern hold within each season, or is it a seasonal
# confound (dry season = low RH + different source mix)?

fig, axes = plt.subplots(3, 3, figsize=(18, 16))

for col_idx, (x_col, y_col, x_label, y_label) in enumerate(PAIRS_FOR_RH):
    for row_idx, season in enumerate(SEASONS_ORDER):
        ax = axes[row_idx, col_idx]
        season_rh = df_rh[df_rh['season'] == season].copy()

        slopes_lo, slopes_hi = [], []
        n_lo_list, n_hi_list = [], []

        for rh_thresh in rh_thresholds:
            lo = season_rh[season_rh['rhum_mean'] <= rh_thresh].dropna(subset=[x_col, y_col])
            hi = season_rh[season_rh['rhum_mean'] > rh_thresh].dropna(subset=[x_col, y_col])
            n_lo_list.append(len(lo))
            n_hi_list.append(len(hi))

            if len(lo) >= 5:
                s, _, r, _, _ = stats.linregress(lo[x_col], lo[y_col])
                slopes_lo.append(s)
            else:
                slopes_lo.append(np.nan)

            if len(hi) >= 5:
                s, _, r, _, _ = stats.linregress(hi[x_col], hi[y_col])
                slopes_hi.append(s)
            else:
                slopes_hi.append(np.nan)

        ax.plot(rh_thresholds, slopes_lo, 'o-', color='#E67E22', linewidth=2,
                markersize=5, label='RH $\leq$ threshold')
        ax.plot(rh_thresholds, slopes_hi, 's-', color='#3498DB', linewidth=2,
                markersize=5, label='RH > threshold')
        ax.axhline(y=1.0, color='gray', linestyle=':', alpha=0.5)

        # Annotate sample sizes at 40% and 60%
        for rh_t_ann in [40, 60]:
            if rh_t_ann in list(rh_thresholds):
                i_ann = list(rh_thresholds).index(rh_t_ann)
                ax.annotate(f'n={n_lo_list[i_ann]}', (rh_t_ann, slopes_lo[i_ann]),
                           textcoords='offset points', xytext=(0, 8),
                           fontsize=7, color='#E67E22', ha='center')
                ax.annotate(f'n={n_hi_list[i_ann]}', (rh_t_ann, slopes_hi[i_ann]),
                           textcoords='offset points', xytext=(0, -12),
                           fontsize=7, color='#3498DB', ha='center')

        ax.set_xlabel('RH Threshold (%)' if row_idx == 2 else '', fontsize=FONT['axis'] - 1)
        ax.set_ylabel('Slope' if col_idx == 0 else '', fontsize=FONT['axis'] - 1)
        ax.grid(True, alpha=0.3)

        if row_idx == 0:
            pair_title = f'{y_label.split("(")[0].strip()}\nvs {x_label.split("(")[0].strip()}'
            ax.set_title(pair_title, fontsize=FONT['title'] - 1)
        if col_idx == 0:
            ax.text(-0.25, 0.5, season, transform=ax.transAxes,
                   fontsize=FONT['title'], fontweight='bold', rotation=90,
                   ha='center', va='center', color=SEASON_COLORS[season])
        if row_idx == 0 and col_idx == 2:
            ax.legend(fontsize=FONT['legend'] - 1, loc='upper right')

plt.suptitle('Season-Stratified RH Threshold Sweep\n'
             '(Does the RH\u2013slope pattern hold within each season?)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'rh_sweep_by_season.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Summary: within-season slope differences at 50% threshold
print('\n' + '=' * 80)
print('WITHIN-SEASON SLOPE COMPARISON AT RH = 50%')
print('=' * 80)
rh_50 = 50
for x_col, y_col, x_label, y_label in PAIRS_FOR_RH:
    pair_name = f'{y_label.split("(")[0].strip()} vs {x_label.split("(")[0].strip()}'
    print(f'\n--- {pair_name} ---')
    print(f'{"Season":<25s} {"Slope_low":>10s} {"Slope_high":>11s} {"n_low":>6s} {"n_high":>7s}')
    print('-' * 65)
    for season in SEASONS_ORDER:
        sdf = df_rh[df_rh['season'] == season]
        lo = sdf[sdf['rhum_mean'] <= rh_50].dropna(subset=[x_col, y_col])
        hi = sdf[sdf['rhum_mean'] > rh_50].dropna(subset=[x_col, y_col])
        s_lo = stats.linregress(lo[x_col], lo[y_col])[0] if len(lo) >= 5 else np.nan
        s_hi = stats.linregress(hi[x_col], hi[y_col])[0] if len(hi) >= 5 else np.nan
        s_lo_str = f'{s_lo:.3f}' if not np.isnan(s_lo) else 'n/a'
        s_hi_str = f'{s_hi:.3f}' if not np.isnan(s_hi) else 'n/a'
        print(f'{season:<25s} {s_lo_str:>10s} {s_hi_str:>11s} {len(lo):6d} {len(hi):7d}')

---

# Revision 3: Temperature Analysis (Biomass Burning Focus)

- Plot **T_mean, T_min, T_max** vs concentration for **biomass burning sources only** (charcoal + wood)
- Investigate and flag **zero-concentration data points**

---

In [ ]:
# =============================================================================
# 3A: Investigate zero-concentration data points
# =============================================================================

bb_conc_cols = ['charcoal_conc', 'wood_conc']
available_bb = [c for c in bb_conc_cols if c in df.columns]

print('Zero-concentration investigation (biomass sources):')
print('=' * 70)

for col in available_bb:
    vals = df[col].dropna()
    n_zero = (vals == 0).sum()
    n_near_zero = (vals < 0.001).sum()
    n_negative = (vals < 0).sum()
    print(f'\n{col}:')
    print(f'  Total non-null: {len(vals)}')
    print(f'  Exactly zero:   {n_zero}')
    print(f'  Near-zero (<0.001): {n_near_zero}')
    print(f'  Negative:       {n_negative}')
    print(f'  Min: {vals.min():.6f}, 1st pct: {vals.quantile(0.01):.6f}')

if 'biomass_conc' in df.columns:
    bb = df['biomass_conc'].dropna()
    n_zero_bb = (bb == 0).sum()
    n_near_zero_bb = (bb < 0.001).sum()
    print(f'\nbiomass_conc (charcoal + wood):')
    print(f'  Exactly zero: {n_zero_bb}')
    print(f'  Near-zero (<0.001): {n_near_zero_bb}')

    # Show dates of zero-concentration points
    zero_rows = df[df['biomass_conc'] == 0][['date', 'charcoal_conc', 'wood_conc',
                                              'biomass_conc', 'kf_total', 'season']]
    if len(zero_rows) > 0:
        print(f'\nZero biomass_conc samples ({len(zero_rows)}):')        
        print(zero_rows.to_string(index=False))
    else:
        print('\nNo exactly-zero biomass_conc samples.')

print('\n\u2192 Zero-conc points will be excluded from temperature regressions below.')

In [ ]:
# =============================================================================
# 3B: T_mean, T_min, T_max vs Biomass Burning Concentration
# =============================================================================

temp_vars = [
    ('tavg', 'T mean (\u00b0C)'),
    ('tmin', 'T min (\u00b0C)'),
    ('tmax', 'T max (\u00b0C)'),
]

bb_cols_plot = [
    ('charcoal_conc', 'Charcoal (\u00b5g/m\u00b3)', SOURCE_CATEGORIES['charcoal']['color']),
    ('wood_conc',     'Wood Burning (\u00b5g/m\u00b3)', SOURCE_CATEGORIES['wood']['color']),
    ('biomass_conc',  'Biomass Total (\u00b5g/m\u00b3)', GROUPED_SOURCE_CATEGORIES['biomass_burning']['color']),
]

fig, axes = plt.subplots(3, 3, figsize=(18, 16))

for row_idx, (t_col, t_label) in enumerate(temp_vars):
    for col_idx, (conc_col, conc_label, clr) in enumerate(bb_cols_plot):
        ax = axes[row_idx, col_idx]

        # Exclude zero-concentration points
        plot_df = df.dropna(subset=[t_col, conc_col]).copy()
        n_before = len(plot_df)
        plot_df = plot_df[plot_df[conc_col] > 0]
        n_excluded = n_before - len(plot_df)

        if len(plot_df) < 5:
            ax.text(0.5, 0.5, 'Insufficient data',
                    transform=ax.transAxes, ha='center')
            continue

        # Color by season
        for season in SEASONS_ORDER:
            mask = plot_df['season'] == season
            subset = plot_df[mask]
            if len(subset) > 0:
                ax.scatter(subset[t_col], subset[conc_col],
                          c=SEASON_COLORS[season], s=40, alpha=0.7,
                          edgecolors='black', linewidths=0.3,
                          label=season if col_idx == 0 and row_idx == 0 else '')

        # Overall regression
        slope, intercept, r, p, _ = stats.linregress(plot_df[t_col], plot_df[conc_col])
        x_line = np.linspace(plot_df[t_col].min(), plot_df[t_col].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=clr,
               linewidth=2.5, alpha=0.8)

        sign = '+' if intercept >= 0 else '-'
        excluded_str = f'\n({n_excluded} zero-conc excluded)' if n_excluded > 0 else ''
        ax.text(0.05, 0.95,
                f'y = {slope:.3f}x {sign} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, p = {p:.3f}\n'
                f'n = {len(plot_df)}{excluded_str}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'] - 1,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        ax.set_xlabel(t_label, fontsize=FONT['axis'] - 1)
        ax.set_ylabel(conc_label if col_idx == 0 else '', fontsize=FONT['axis'] - 1)
        if row_idx == 0:
            ax.set_title(conc_label, fontsize=FONT['title'] - 1, fontweight='bold')
        ax.grid(True, alpha=0.3)

# Shared legend from first row
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3,
           fontsize=FONT['legend'], bbox_to_anchor=(0.5, 1.02))

plt.suptitle('Temperature vs Biomass Burning Source Concentration\n(zero-concentration points excluded)',
             fontsize=FONT['title'] + 1, fontweight='bold', y=1.06)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'temp_vs_biomass_concentration.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

# Revision 4: Relative Humidity × Season × Biomass Burning

Source concentration vs **absolute RH** by season (Belg, Kiremt, Dry kept separate), focused on biomass burning.

---

In [ ]:
# =============================================================================
# 4: Source Concentration vs Absolute RH by Season (Biomass Focus)
# =============================================================================

bb_sources = [
    ('charcoal_conc', 'Charcoal', SOURCE_CATEGORIES['charcoal']['color'],
     SOURCE_CATEGORIES['charcoal']['marker']),
    ('wood_conc', 'Wood Burning', SOURCE_CATEGORIES['wood']['color'],
     SOURCE_CATEGORIES['wood']['marker']),
    ('biomass_conc', 'Biomass Total', GROUPED_SOURCE_CATEGORIES['biomass_burning']['color'], 'o'),
]

fig, axes = plt.subplots(len(bb_sources), 3, figsize=(18, 5 * len(bb_sources)))

for row_idx, (conc_col, conc_label, clr, mkr) in enumerate(bb_sources):
    for col_idx, season in enumerate(SEASONS_ORDER):
        ax = axes[row_idx, col_idx]
        season_df = df[(df['season'] == season)].dropna(
            subset=['rhum_mean', conc_col]).copy()
        season_df = season_df[season_df[conc_col] > 0]  # exclude zeros

        if len(season_df) < 5:
            ax.text(0.5, 0.5, f'{season}\nInsufficient data',
                    transform=ax.transAxes, ha='center', va='center')
            continue

        ax.scatter(season_df['rhum_mean'], season_df[conc_col],
                  c=SEASON_COLORS[season], marker=mkr, s=50, alpha=0.7,
                  edgecolors='black', linewidths=0.5)

        # Regression
        slope, intercept, r, p, _ = stats.linregress(
            season_df['rhum_mean'], season_df[conc_col])
        x_line = np.linspace(season_df['rhum_mean'].min(),
                             season_df['rhum_mean'].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=clr,
               linewidth=2, alpha=0.8)

        sign = '+' if intercept >= 0 else '-'
        ax.text(0.05, 0.95,
                f'y = {slope:.4f}x {sign} {abs(intercept):.3f}\n'
                f'R\u00b2 = {r**2:.3f}, p = {p:.3f}\nn = {len(season_df)}',
                transform=ax.transAxes, va='top', fontsize=FONT['annot'] - 1,
                bbox=dict(boxstyle='round', facecolor=SEASON_COLORS[season], alpha=0.15))

        ax.set_xlabel('Daily Mean RH (%)', fontsize=FONT['axis'] - 1)
        if col_idx == 0:
            ax.set_ylabel(f'{conc_label} (\u00b5g/m\u00b3)', fontsize=FONT['axis'])
        if row_idx == 0:
            ax.set_title(season, fontsize=FONT['title'],
                        color=SEASON_COLORS[season], fontweight='bold')
        ax.grid(True, alpha=0.3)

plt.suptitle('Biomass Burning Concentration vs Relative Humidity by Season',
             fontsize=FONT['title'] + 2, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'rh_vs_biomass_by_season.png'),
            dpi=200, bbox_inches='tight')
plt.show()

---

# Revision 5: Seasonal Correlation Matrices

Three separate met × source correlation heatmaps, one per season.

---

In [ ]:
# =============================================================================
# 5: Seasonal Correlation Matrices (Met x Source)
# =============================================================================

met_vars = ['tavg', 'tmin', 'tmax', 'temp_range', 'prcp', 'wspd_mean', 'rhum_mean', 'pres']
source_vars = [c for c in conc_cols + ['kf_total', 'biomass_conc', 'fossil_comb_conc']
               if c in df.columns]
met_available = [c for c in met_vars if c in df.columns]
corr_cols_all = met_available + source_vars

met_rename = {
    'tavg': 'T avg', 'tmin': 'T min', 'tmax': 'T max',
    'temp_range': '\u0394T (Tmax\u2212Tmin)', 'prcp': 'Precipitation',
    'wspd_mean': 'Wind Speed', 'rhum_mean': 'Rel. Humidity', 'pres': 'Pressure'
}
source_rename = {
    'charcoal_conc': 'Charcoal', 'wood_conc': 'Wood',
    'fossil_fuel_conc': 'Fossil Fuel', 'polluted_marine_conc': 'Poll. Marine',
    'sea_salt_conc': 'Sea Salt', 'kf_total': 'Total KF',
    'biomass_conc': 'Biomass (grp)', 'fossil_comb_conc': 'Fossil (grp)',
}

fig, axes = plt.subplots(1, 3, figsize=(24, 7))

for idx, season in enumerate(SEASONS_ORDER):
    ax = axes[idx]
    season_df = df[df['season'] == season][corr_cols_all].dropna()
    n_samples = len(season_df)

    if n_samples < 10:
        ax.text(0.5, 0.5, f'{season}\nn={n_samples} (too few)',
                transform=ax.transAxes, ha='center', va='center',
                fontsize=FONT['title'])
        ax.set_title(season, fontsize=FONT['title'],
                    color=SEASON_COLORS[season], fontweight='bold')
        continue

    corr_matrix = season_df.corr()
    corr_block = corr_matrix.loc[met_available, source_vars]
    corr_block = corr_block.rename(index=met_rename, columns=source_rename)

    sns.heatmap(corr_block, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-0.8, vmax=0.8, linewidths=0.5, ax=ax,
                annot_kws={'fontsize': FONT['annot'] - 1},
                cbar_kws={'shrink': 0.8})

    ax.set_title(f'{season}\n(n={n_samples})', fontsize=FONT['title'],
                color=SEASON_COLORS[season], fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right',
                       fontsize=FONT['tick'] - 1)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=FONT['tick'] - 1)

plt.suptitle('Meteorology \u00d7 Source Concentration Correlation by Season',
             fontsize=FONT['title'] + 2, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig(os.path.join(dirs['plots'], 'seasonal_correlation_matrices.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Print sample sizes
print('\nSample sizes per season (complete cases for correlation):')
for season in SEASONS_ORDER:
    n = len(df[df['season'] == season][corr_cols_all].dropna())
    print(f'  {season}: n={n}')